# Hair ‘양호’ 원본과 기존 Hair 데이터 교차 검사

이 노트북은 **학습하거나 데이터셋을 만들지 않습니다.**
Drive의 기존 Hair Clean ZIP과 새 양호 ZIP을 읽고, 동일 사진·동일 사람 ID가 있는지 확인합니다.
두 ZIP을 수정하지 않으며 결과는 `mediflow_Project/2_results/hair/` 아래에
새 폴더와 ZIP으로 저장합니다.

현재 양호 ZIP에는 JPG 534장이 있으며 라벨 JSON은 없습니다.
여섯 증상값이 모두 0이라는 것은 사용자가 원본에서 확인한 내용으로 기록하고,
이 노트북이 파일별 라벨을 독립 검증했다고 표현하지 않습니다.
기존 Hair v1 원본만 비교하며 저장 증강본은 다시 분할할 v2 원본이 아닙니다.

**실행:** Colab에서 이 노트북을 Drive의 `notebooks`에 업로드해 열고,
GPU 없이 셀을 순서대로 실행하세요. 기존 Hair ZIP 약 3GB를 Colab 작업 공간에
복사하므로 빈 공간과 시간이 필요합니다. 결과의 `audit_summary.json` 및
`good_cross_matches.csv`를 확인한 후 ZIP을 공유하면 다음 데이터셋 구성으로 이어갈 수 있습니다.


<!-- mediflow-notebook-guide-v1 -->
### 실행 안내

- **이 노트북의 목적:** 새 Hair ‘양호’ 사진과 기존 Hair 사진의 중복을 검사합니다.
- **실행 순서:** Colab에서 첫 셀부터 아래로 차례대로 실행합니다. 중간 셀만 단독 실행하면 앞에서 만든 설정과 코드가 없어 오류가 날 수 있습니다.
- **수정할 곳:** 앞부분의 경로·실험 설정 셀을 먼저 확인합니다. 내장 실행 코드 셀은 펼쳐서 읽을 수 있지만 보통 수정하지 않습니다.
- **결과 확인:** 끝부분의 저장 경로와 보고서 출력에서 결과를 확인합니다. ZIP과 학습 결과를 혼동하지 않도록 각 단계의 설명을 읽어 주세요.


<!-- mediflow-notebook-sections-v1 -->
## 1. Drive 연결과 검사 대상 지정


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil
import zipfile
from datetime import datetime, timezone
from uuid import uuid4

PROJECT_ROOT = Path('/content/drive/MyDrive/mediflow_Project')
DATASET_DIR = PROJECT_ROOT / 'datasets'
hair_candidates = [
    DATASET_DIR / 'hair_datasets',
    DATASET_DIR / 'hair_datasets.zip',
]
valid_hair_zips = [p for p in hair_candidates if p.is_file() and zipfile.is_zipfile(p)]
if len(valid_hair_zips) != 1:
    available = sorted(p.name for p in DATASET_DIR.iterdir()) if DATASET_DIR.is_dir() else []
    raise FileNotFoundError(
        '기존 Hair ZIP을 하나로 확인할 수 없습니다. '
        f'확인한 후보: {[str(p) for p in hair_candidates]} / '
        f'datasets 폴더 항목: {available}'
    )
OLD_ZIP = valid_hair_zips[0]
GOOD_ZIP = DATASET_DIR / 'hair_good_raw_v1.zip'

for path in (OLD_ZIP, GOOD_ZIP):
    if not path.is_file() or not zipfile.is_zipfile(path):
        raise FileNotFoundError(f'ZIP 파일을 확인하세요: {path}')
    print(path.name, f'{path.stat().st_size / 1024**2:.1f} MiB')

required = OLD_ZIP.stat().st_size + GOOD_ZIP.stat().st_size + 1024**3
if shutil.disk_usage('/content').free < required:
    raise RuntimeError('Colab 작업 공간이 부족합니다. 새 런타임을 사용하세요.')

work = Path('/content') / ('hair_good_audit_' + uuid4().hex[:8])
work.mkdir()
local_old = work / 'hair_datasets.zip'
local_good = work / 'hair_good_raw_v1.zip'
shutil.copyfile(OLD_ZIP, local_old)
shutil.copyfile(GOOD_ZIP, local_good)
print('Colab 임시 작업 공간으로 두 ZIP 복사 완료:', work)


## 2. 검사 함수

아래 코드는 이 노트북에 포함돼 있습니다. 별도의 Python 파일을 Drive에 올릴 필요가 없습니다.


In [ ]:
"""Read-only cross audit of Hair clean v1 and user-confirmed all-zero scalp images."""

from __future__ import annotations

import csv
import hashlib
import io
import json
import zipfile
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath

from PIL import Image

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLITS = {"train", "val", "test"}


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _subject_id(name: str) -> str | None:
    first = PurePosixPath(name).name.split("_", 1)[0]
    return first if first.isdecimal() else None


def _old_label(path: PurePosixPath) -> tuple[str, str] | None:
    parts = path.parts
    for index, part in enumerate(parts):
        if part.lower() == "original" and len(parts) > index + 3:
            split = parts[index + 1].lower()
            if split in SPLITS:
                return split, parts[index + 2]
    return None


def _scan(path: Path, *, existing: bool) -> tuple[list[dict], dict]:
    records = []
    skipped = Counter()
    seen_paths = set()
    with zipfile.ZipFile(path) as archive:
        for member in archive.infolist():
            if member.is_dir():
                continue
            name = member.filename
            parts = PurePosixPath(name)
            if parts.is_absolute() or ".." in parts.parts or "\\" in name:
                raise ValueError(f"Unsafe ZIP member: {name}")
            if name.casefold() in seen_paths:
                raise ValueError(f"Repeated ZIP path: {name}")
            seen_paths.add(name.casefold())
            if parts.suffix.lower() not in IMAGE_SUFFIXES:
                skipped["non_image"] += 1
                continue
            label = _old_label(parts) if existing else None
            if existing and label is None:
                skipped["not_original"] += 1
                continue
            data = archive.read(member)
            with Image.open(io.BytesIO(data)) as image:
                rgb = image.convert("RGB")
                pixels = hashlib.sha256()
                pixels.update(f"{rgb.width}x{rgb.height}:".encode("ascii"))
                pixels.update(rgb.tobytes())
                size = [rgb.width, rgb.height]
            row = {
                "path": name,
                "subject_id": _subject_id(name),
                "byte_sha256": hashlib.sha256(data).hexdigest(),
                "pixel_sha256": pixels.hexdigest(),
                "size": size,
            }
            if existing:
                row["split"], row["class_name"] = label
            records.append(row)
            if len(records) % 2000 == 0:
                print(f"{'Hair v1' if existing else '양호'}: {len(records)}장 확인")
    if not records:
        raise ValueError(f"No {'original split' if existing else 'good'} images: {path}")
    return records, dict(skipped)


def _write_csv(path: Path, rows: list[dict], fields: list[str]) -> None:
    with path.open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def audit(existing_zip: Path, good_zip: Path, output_dir: Path) -> dict:
    """Inspect the two ZIPs without changing either input or making train/test splits."""
    existing_zip = Path(existing_zip).resolve()
    good_zip = Path(good_zip).resolve()
    output_dir = Path(output_dir).resolve()
    if existing_zip == good_zip:
        raise ValueError("기존 Hair ZIP과 양호 ZIP을 각각 지정하세요.")
    if output_dir.exists():
        raise FileExistsError(f"보고서 폴더가 이미 있습니다: {output_dir}")
    if not zipfile.is_zipfile(existing_zip) or not zipfile.is_zipfile(good_zip):
        raise ValueError("두 입력 모두 ZIP 파일이어야 합니다.")

    old, old_skipped = _scan(existing_zip, existing=True)
    good, good_skipped = _scan(good_zip, existing=False)
    old_bytes = defaultdict(list)
    old_pixels = defaultdict(list)
    old_subjects = defaultdict(list)
    for row in old:
        old_bytes[row["byte_sha256"]].append(row)
        old_pixels[row["pixel_sha256"]].append(row)
        if row["subject_id"] is not None:
            old_subjects[row["subject_id"]].append(row)

    details = []
    for row in good:
        byte_matches = old_bytes[row["byte_sha256"]]
        pixel_matches = old_pixels[row["pixel_sha256"]]
        subject_matches = old_subjects[row["subject_id"]] if row["subject_id"] else []
        details.append(
            {
                "good_path": row["path"],
                "subject_id": row["subject_id"] or "",
                "byte_matches": len(byte_matches),
                "pixel_matches": len(pixel_matches),
                "subject_matches": len(subject_matches),
                "old_splits_for_subject": "|".join(sorted({m["split"] for m in subject_matches})),
                "old_classes_for_subject": "|".join(
                    sorted({m["class_name"] for m in subject_matches})
                ),
                "exact_old_paths": "|".join(
                    sorted({m["path"] for m in byte_matches + pixel_matches})
                ),
            }
        )

    good_people = {r["subject_id"] for r in good if r["subject_id"] is not None}
    old_people = {r["subject_id"] for r in old if r["subject_id"] is not None}
    summary = {
        "protocol": "hair_good_cross_audit_v1",
        "status": "read_only_audit_not_a_v2_dataset",
        "existing_zip_name": existing_zip.name,
        "existing_zip_sha256": sha256_file(existing_zip),
        "good_zip_name": good_zip.name,
        "good_zip_sha256": sha256_file(good_zip),
        "existing_original_images": len(old),
        "existing_split_counts": dict(Counter(row["split"] for row in old)),
        "existing_class_counts": dict(Counter(row["class_name"] for row in old)),
        "good_images": len(good),
        "good_subject_ids": len(good_people),
        "existing_subject_ids": len(old_people),
        "good_images_without_subject_id": sum(r["subject_id"] is None for r in good),
        "existing_images_without_subject_id": sum(r["subject_id"] is None for r in old),
        "shared_subject_ids": len(good_people & old_people),
        "good_images_matching_old_bytes": sum(row["byte_matches"] > 0 for row in details),
        "good_images_matching_old_pixels": sum(row["pixel_matches"] > 0 for row in details),
        "good_images_with_old_subject": sum(row["subject_matches"] > 0 for row in details),
        "old_skipped": old_skipped,
        "good_skipped": good_skipped,
        "good_label_source": "user_confirmed_all_six_values_zero; no_json_in_good_zip",
        "limits": [
            "A shared subject ID is a grouping concern, not proof of identical images.",
            "Exact byte/pixel hashes do not exclude near duplicates.",
            "Subject matching uses the first numeric filename component; "
            "unmatched IDs remain unverified.",
            "No train/validation/test split or augmentation was created.",
        ],
    }
    output_dir.mkdir(parents=True)
    (output_dir / "audit_summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    _write_csv(
        output_dir / "good_cross_matches.csv",
        details,
        [
            "good_path",
            "subject_id",
            "byte_matches",
            "pixel_matches",
            "subject_matches",
            "old_splits_for_subject",
            "old_classes_for_subject",
            "exact_old_paths",
        ],
    )
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary


## 3. 교차 검사 실행·결과 저장

기존 ZIP은 읽기 전용이며 결과는 매번 새 이름으로 보존합니다.


In [ ]:
parent = PROJECT_ROOT / '2_results' / 'hair'
parent.mkdir(parents=True, exist_ok=True)
run_name = (
    'hair_good_cross_audit_'
    + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    + '_'
    + uuid4().hex[:8]
)
report_dir = parent / run_name
summary = audit(local_old, local_good, report_dir)
report_zip = Path(shutil.make_archive(str(report_dir), 'zip', root_dir=parent, base_dir=run_name))
print('검사 보고서:', report_dir)
print('공유할 결과 ZIP:', report_zip)
print('이 노트북은 Train/Validation/Test를 만들거나 학습하지 않았습니다.')
